In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
from torchvision.utils import save_image, make_grid
import os

batch_size = 128
z_dim = 100 # Size of noise vector
image_size = 28
channels = 1
epochs = 50
lr = 0.0002
beta1 = 0.5

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

os.makedirs("generated_images", exist_ok=True)

In [2]:
transform = transforms.Compose([
    transforms.Resize(image_size),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

train_loader = torch.utils.data.DataLoader(
    datasets.MNIST('.', train=True, download=True, transform=transform),
    batch_size=batch_size,
    shuffle=True
)

100%|██████████| 9.91M/9.91M [01:44<00:00, 94.5kB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 80.5kB/s]
100%|██████████| 1.65M/1.65M [00:07<00:00, 220kB/s] 
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.00MB/s]


In [3]:
class Generator(nn.Module):
    def __init__(self, z_dim):
        super(Generator, self).__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(z_dim, 256, 7, 1, 0, bias=False),
            nn.BatchNorm2d(256),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(256, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.ReLU(True),
            
            nn.ConvTranspose2d(128, 1, 4, 2, 1, bias=False),
            nn.Tanh()
        )
    
    def forward(self, z):
        return self.net(z)

In [4]:
class Discriminator(nn.Module):
    def __init__(self):
        super(Discriminator, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(1, 64, 4, 2, 1, bias=False),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Conv2d(64, 128, 4, 2, 1, bias=False),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(0.2, inplace=True),
            
            nn.Flatten(),
            nn.Linear(128 * 7 * 7, 1),
            nn.Sigmoid()
        )
    
    def forward(self, img):
        return self.net(img)

In [5]:
generator = Generator(z_dim).to(device)
discriminator = Discriminator().to(device)

criterion = nn.BCELoss()

optimizer_G = optim.Adam(generator.parameters(), lr=lr, betas=(beta1, 0.999))
optimizer_D = optim.Adam(discriminator.parameters(), lr=lr, betas=(beta1, 0.999))

In [6]:
def generate_and_save_images(epoch):
    generator.eval()
    with torch.no_grad():
        z = torch.randn(64, z_dim, 1, 1).to(device)
        fake_imgs = generator(z)
        fake_imgs = fake_imgs * 0.5 + 0.5
        save_image(fake_imgs, f"generated_images/sample_epoch_{epoch}.png", nrow=8)
    generator.train()

In [7]:
k = 3 # Generator updates per iteration
p = 1

In [9]:
for epoch in range(1, epochs + 1):
    for i, (real_imgs, _) in enumerate(train_loader):
        batch_size_curr = real_imgs.size(0)
        real_imgs = real_imgs.to(device)
        
        real = torch.ones(batch_size_curr, 1, device=device)
        fake = torch.zeros(batch_size_curr, 1, device=device)
        
        for _ in range(p):
            z = torch.randn(batch_size_curr, z_dim, 1, 1, device=device)
            fake_imgs = generator(z)
            
            real_validity = discriminator(real_imgs)
            d_real_loss = criterion(real_validity, real)
            
            fake_validity = discriminator(fake_imgs.detach())
            d_fake_loss = criterion(fake_validity, fake)
            
            d_loss = d_real_loss + d_fake_loss
            
            optimizer_D.zero_grad()
            d_loss.backward()
            optimizer_D.step()
        
        for _ in range(k):
            z = torch.randn(batch_size_curr, z_dim, 1, 1, device=device)
            fake_imgs = generator(z)
            
            validity = discriminator(fake_imgs)
            g_loss = criterion(validity, real) # fool D: label as real
            
            optimizer_G.zero_grad()
            g_loss.backward()
            optimizer_G.step()
        
        if i % 200 == 0:
            print(f"[Epoch {epoch}/{epochs}] [Batch {i}/{len(train_loader)}] "
                  f"D Loss: {d_loss.item():.4f} | G Loss: {g_loss.item():.4f}")
    
    generator.eval()
    with torch.no_grad():
        z = torch.randn(64, z_dim, 1, 1, device=device)
        samples = generator(z)
        samples = samples * 0.5 + 0.5
        save_image(samples, f"generated_images/epoch_{epoch}.png", nrow=8)
    generator.train()

[Epoch 1/50] [Batch 0/469] D Loss: 1.1846 | G Loss: 0.7890
[Epoch 1/50] [Batch 200/469] D Loss: 1.1643 | G Loss: 0.5928
[Epoch 1/50] [Batch 400/469] D Loss: 1.1492 | G Loss: 0.8336
[Epoch 2/50] [Batch 0/469] D Loss: 0.5956 | G Loss: 0.6891
[Epoch 2/50] [Batch 200/469] D Loss: 1.0855 | G Loss: 0.7758
[Epoch 2/50] [Batch 400/469] D Loss: 1.0480 | G Loss: 1.3277
[Epoch 3/50] [Batch 0/469] D Loss: 1.1577 | G Loss: 1.0715
[Epoch 3/50] [Batch 200/469] D Loss: 1.1174 | G Loss: 0.8326
[Epoch 3/50] [Batch 400/469] D Loss: 1.1000 | G Loss: 0.9102
[Epoch 4/50] [Batch 0/469] D Loss: 1.0286 | G Loss: 1.0526
[Epoch 4/50] [Batch 200/469] D Loss: 1.1339 | G Loss: 1.1113
[Epoch 4/50] [Batch 400/469] D Loss: 1.0744 | G Loss: 0.9375
[Epoch 5/50] [Batch 0/469] D Loss: 1.2109 | G Loss: 0.4738
[Epoch 5/50] [Batch 200/469] D Loss: 1.1736 | G Loss: 1.2199
[Epoch 5/50] [Batch 400/469] D Loss: 1.1139 | G Loss: 1.2729
[Epoch 6/50] [Batch 0/469] D Loss: 1.2132 | G Loss: 0.6227
[Epoch 6/50] [Batch 200/469] D Loss:

KeyboardInterrupt: 